<a href="https://colab.research.google.com/github/quprep/quprep/blob/main/examples/12_exporters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/quprep/quprep/v0.6.0?labpath=examples%2F12_exporters.ipynb)

# 12 — New Exporters (v0.6.0)

Three new exporters introduced in v0.6.0:

| Exporter | Output | Extra dep needed |
|---|---|---|
| `BraketExporter` | `braket.circuits.Circuit` | `pip install quprep[braket]` |
| `QSharpExporter` | `str` (Q# 1.0 source) | None for generation; `quprep[qsharp]` for Azure Quantum |
| `IQMExporter` | `dict` (PRX+CZ JSON) | None for generation; `quprep[iqm]` for hardware |

Install all extras:
```bash
pip install quprep[braket,qsharp,iqm]
```

In [ ]:
import json

import numpy as np

import quprep as qd
from quprep.encode.zz_feature_map import ZZFeatureMapEncoder

rng = np.random.default_rng(42)
X = rng.uniform(0, 1, size=(5, 3)) * np.pi

enc_angle = qd.AngleEncoder(rotation="ry").encode(np.array([0.3, 1.1, 0.7]))
enc_zz    = ZZFeatureMapEncoder(reps=1).encode(np.array([0.5, 1.2, 0.8]))

## 1. Amazon Braket

Returns a `braket.circuits.Circuit`. Compatible with AWS managed simulators and IonQ, Rigetti, OQC, and other Braket hardware providers.

**Supported encodings:** `angle`, `entangled_angle`, `basis`, `iqp`, `zz_feature_map`, `tensor_product`, `reupload`, `hamiltonian`

In [ ]:
try:
    from quprep.export.braket_export import BraketExporter

    exp_b = BraketExporter()

    # Angle encoding
    circuit = exp_b.export(enc_angle)
    print("Angle encoding:")
    print(circuit)
    print()

    # ZZ feature map (shows CZ gates)
    circuit_zz = exp_b.export(enc_zz)
    print("ZZ feature map:")
    print(circuit_zz)
    print()

    # Batch export
    result = qd.prepare(X, encoding="angle")
    batch = exp_b.export_batch(result.encoded)
    print(f"Batch: {len(batch)} circuits, first circuit depth: {batch[0].depth}")

except ImportError as e:
    print(f"Braket not installed — {e}")

In [ ]:
# Via prepare()
try:
    result2 = qd.prepare(X[:2], encoding="angle", framework="braket")
    print(f"prepare() → braket: {type(result2.circuits[0]).__name__}")
except ImportError as e:
    print(f"Braket not installed — {e}")

## 2. Q# (Microsoft Azure Quantum)

Generates a complete Q# 1.0 source file as a string. No dependencies required for string generation — install `quprep[qsharp]` only for actual Azure Quantum submission.

**Supported encodings:** all 10 encodings (angle, entangled_angle, basis, iqp, zz_feature_map, pauli_feature_map, random_fourier, tensor_product, reupload, hamiltonian)

In [ ]:
from quprep.export.qsharp_export import QSharpExporter

exp_qs = QSharpExporter(namespace="QuPrepDemo", operation_name="EncodeFeatures")

# Angle encoding
qsharp_angle = exp_qs.export(enc_angle)
print("Angle encoding → Q# source:")
print(qsharp_angle)

In [ ]:
# ZZ feature map — shows H, Rz, CNOT gates
qsharp_zz = exp_qs.export(enc_zz)
print("ZZ feature map → Q# source:")
print(qsharp_zz)

In [ ]:
# Custom namespace and operation name
exp_qs2 = QSharpExporter(namespace="MyOrg.Circuits", operation_name="KernelMap")
qsharp_custom = exp_qs2.export(enc_angle)
print(qsharp_custom.splitlines()[0])  # namespace line

# Batch
batch_qs = exp_qs.export_batch([enc_angle, enc_zz])
print(f"\nBatch: {len(batch_qs)} Q# source strings")

# Via prepare()
result_qs = qd.prepare(X[:2], encoding="angle", framework="qsharp")
print(f"prepare() → qsharp: first circuit is str = {isinstance(result_qs.circuits[0], str)}")

## 3. IQM Native Format

Returns a plain Python `dict` matching the IQM circuit JSON schema. No dependencies required for dict generation — install `quprep[iqm]` only for hardware submission via `iqm_client.Circuit.from_dict()`.

**Native gate set:** PRX(angle_t, phase_t) and CZ
- Ry → PRX(θ/2π, 0.25)
- Rx → PRX(θ/2π, 0.0)
- Rz → H · Rx · H (virtual decomposition)
- CZ → native

**Supported encodings:** all 10 encodings

In [ ]:
from quprep.export.iqm_export import IQMExporter

exp_iqm = IQMExporter(circuit_name="feature_map", qubit_prefix="QB")

# Angle encoding
circuit_iqm = exp_iqm.export(enc_angle)
print("Angle encoding (IQM dict):")
print(json.dumps(circuit_iqm, indent=2))

In [ ]:
# ZZ feature map — shows CZ gates from pairwise interactions
circuit_iqm_zz = exp_iqm.export(enc_zz)
gate_names = [op["name"] for op in circuit_iqm_zz["instructions"]]
print(f"ZZ feature map: {len(circuit_iqm_zz['instructions'])} instructions")
print(f"  gate types  : {sorted(set(gate_names))}")
print(f"  CZ count    : {gate_names.count('cz')}")
print()

# JSON-serializable — ready to write to disk or pass to iqm_client
json_str = json.dumps(circuit_iqm)
print(f"JSON bytes: {len(json_str)}")

# Batch
batch_iqm = exp_iqm.export_batch([enc_angle, enc_zz])
print(f"Batch: {len(batch_iqm)} circuit dicts")

# Via prepare()
result_iqm = qd.prepare(X[:2], encoding="angle", framework="iqm")
print(f"prepare() → iqm: first circuit is dict = {isinstance(result_iqm.circuits[0], dict)}")